# Redes e a internet

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx roda o Java com o seu JDK. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Acompanhar um pedido da web pelo DNS, TCP e HTTP, e escolher TCP ou UDP para uma aplicação
- Verificar se um endereço IPv4 é válido e lê-lo como um único número de 32 bits
- Achar os endereços de rede, de broadcast e de hosts de um endereço com prefixo
- Escolher o próximo salto de um pacote pelo maior prefixo correspondente, como um roteador faz

## Em resumo

A internet nunca manda um arquivo de uma vez. Ela corta os dados em **pacotes** de cerca de 1500 bytes, cada um com um **cabeçalho** (de onde vem e para onde vai) e um pedaço dos dados. Os **roteadores** passam cada pacote um salto mais perto do destino; os pacotes podem seguir caminhos diferentes e chegar fora de ordem.

**Cortando um arquivo em pacotes:**

In [ ]:
int dados = 1460; // bytes de dados num pacote típico de 1500 bytes, depois de 40 bytes de cabeçalhos
for (long tamanho : new long[] {1_000, 100_000, 5_000_000}) {
    long pacotes = (tamanho + dados - 1) / dados; // arredondado para cima
    IO.println(String.format("%,10d bytes: %,6d pacotes", tamanho, pacotes));
}

Todo dispositivo tem um **endereço IP**: 4 bytes escritos como `192.168.1.130`, que para o computador são um único número de 32 bits. Um **prefixo** como `/26` diz quantos desses bits identificam a **rede**; o resto numera os **hosts** dentro dela.

![O endereço 192.168.1.130/26 dividido em bits de rede e de host](prefix.pt-br.svg)

Todos os bits de host 0 é o **endereço de rede**, todos 1 é o **endereço de broadcast**, e os endereços do meio são para os dispositivos. Um roteador escolhe para onde mandar um pacote pelo **maior prefixo** da sua tabela que corresponde ao destino.

**Quantos endereços cada prefixo dá?**

In [ ]:
for (int prefixo = 22; prefixo <= 30; prefixo++) {
    long enderecos = 1L << (32 - prefixo);
    IO.println(String.format("/%d: %,5d endereços, %,5d para dispositivos", prefixo, enderecos, enderecos - 2));
}

Por cima do IP, o **TCP** numera os dados, confirma o que chegou e reenvia o que se perdeu, para que páginas e arquivos cheguem completos e em ordem. O **UDP** pula tudo isso, o que serve para chamadas de vídeo e jogos, em que um pacote atrasado não serve para nada. Antes de conectar, seu computador pergunta ao **DNS** o endereço de um nome como `example.com`. Depois o navegador abre uma conexão TCP e manda um pedido **HTTP**: `GET /index.html HTTP/1.1`.

## Leituras

- [The Internet](https://www.youtube.com/watch?v=AEaKrq3SpW8) (Crash Course Computer Science #29, 12 min): Como uma mensagem vira pacotes, como os roteadores os passam adiante salto a salto, por que o TCP numera e confirma os pacotes e o UDP não, e o que o DNS faz antes de tudo isso.
- [How DNS Works](https://howdns.works/) (DNSimple (a comic in 6 short episodes), 15 min): Acompanhe um nome desde o cache do navegador até o resolvedor, depois os servidores raiz, de TLD e autoritativo, e repare onde cada resposta fica em cache para a próxima vez.
- [Subnet Cheat Sheet](https://www.freecodecamp.org/news/subnet-cheat-sheet-24-subnet-mask-30-26-27-29-and-other-ip-address-cidr-network-references/) (freeCodeCamp, 9 min): Como um prefixo como /26 divide um endereço em bits de rede e de host, como uma máscara em binário marca a parte de rede, e a tabela de prefixos com os endereços utilizáveis (/26 tem 62). As máscaras curinga e o caso especial do /31 são mais do que você precisa.

## Verifique

1. Por que uma chamada de vídeo usa UDP enquanto um download usa TCP?
2. Quais são os endereços de rede e de broadcast de `10.20.30.40/24`?
3. Um roteador tem rotas para `10.0.0.0/8` e `10.1.0.0/16`. Para onde ele manda um pacote para `10.1.2.3`, e por quê?

## Desafios

Resolva os desafios da unidade aqui mesmo: cada célula começa com `// sphinx: challenge …`, e rodá-la roda **todos** os testes do desafio, inclusive os ocultos. Passar conta no seu progresso na barra lateral do Sphinx, como o Submit. A descrição completa, com exemplos e dicas, abre pela barra lateral.

### Endereço IPv4 válido · Fácil

In [ ]:
// sphinx: challenge valid-ipv4
void main() {
    String address = IO.readln().trim(); // por exemplo 192.168.1.1

    // TODO: separe nos pontos, confira cada regra e imprima Valid ou Invalid
}

### Lendo um pedido HTTP · Fácil

In [ ]:
// sphinx: challenge http-request
void main() {
    String requestLine = IO.readln(); // por exemplo GET /index.html HTTP/1.1
    String line = IO.readln();
    while (!line.isEmpty()) {
        // TODO: confira a linha de cabeçalho e guarde o Host

        line = IO.readln();
    }
    // TODO: confira a linha de pedido, depois imprima as quatro linhas ou 400 Bad Request
}

### Calculadora de sub-rede · Médio

In [ ]:
// sphinx: challenge subnet-calculator
void main() {
    String[] input = IO.readln().trim().split("/"); // por exemplo ["192.168.1.130", "26"]
    String[] parts = input[0].split("\\.");
    int prefix = Integer.parseInt(input[1]);

    // TODO: transforme o endereço num único número, ache o bloco em que ele está e imprima as cinco linhas
}

### Tabela de roteamento · Difícil

In [ ]:
// sphinx: challenge routing-table
void main() {
    int r = Integer.parseInt(IO.readln().trim());
    String[] routes = new String[r];
    for (int i = 0; i < r; i++) {
        routes[i] = IO.readln().trim(); // por exemplo 10.0.0.0/8 Office
    }
    int n = Integer.parseInt(IO.readln().trim());
    for (int i = 0; i < n; i++) {
        String address = IO.readln().trim();

        // TODO: ache a rota que corresponde com o maior prefixo e imprima endereco -> proximo salto
    }
}

### Teste: Inspetor de pacotes · Difícil

In [ ]:
// sphinx: challenge packet-inspector
void main() {
    String[] hex = IO.readln().trim().split(" ");
    int[] bytes = new int[hex.length];
    for (int i = 0; i < hex.length; i++) {
        bytes[i] = Integer.parseInt(hex[i], 16);
    }

    // TODO: leia os campos do cabeçalho IPv4 em bytes, confira o checksum e depois a parte UDP
}